# 05 — 主题出现与类型分析

**BWV861 音乐形式化分析系统** · Step 5

构建主题出现 {1D52C}_q，划分严格主题类型 T_i 和对称族 F_ℓ。

- ∼_strict: κ_a == κ_b → T_i
- ∼_sym: μ(T_i) == μ(T_j) → F_ℓ

In [1]:
import sys, os
PROJECT_ROOT = r"e:\大三\Cline Projects\bien_project"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from music_analysis.features import load_features
from music_analysis.motifs import load_motifs
from music_analysis.themes import (
    build_theme_occurrences, build_strict_types,
    build_symmetric_families, save_themes,
)

features, single_windows, multi_windows = load_features()
motifs = load_motifs()
print(f"✅ 已加载 {len(features)} 条特征, {len(motifs)} 个动机原型")

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.

✅ 已加载 1733 条特征, 736 个动机原型


In [2]:
# 构建主题出现
occurrences = build_theme_occurrences(features, motifs)

# 严格主题类型
strict_types = build_strict_types(occurrences)

# 对称族
symmetric_families = build_symmetric_families(strict_types)

save_themes(occurrences, strict_types, symmetric_families)

# 摘要
print(f"\n总结: Q={len(occurrences)} 次主题出现, "
      f"K={len(strict_types)} 个严格类型, "
      f"L={len(symmetric_families)} 个对称族")

构建了 1025 次主题出现 (单声部)
严格主题类型数 K = 736
  出现次数: min=1, max=11, mean=1.4
  Singleton 类型 (仅出现1次): 626
对称族数 L = 736
  每族严格类型数: min=1, max=1, mean=1.0
  含多个严格类型的族: 0
✅ 主题数据已保存: data/themes.pkl

总结: Q=1025 次主题出现, K=736 个严格类型, L=736 个对称族


## 5.1 压缩增益过滤

按压缩增益 $ \widehat{G}_i > \theta_G $ 过滤掉信息量不足的主题类型。

$$G_i = L_{\mathrm{raw}}(T_i) - L_{\mathrm{model}}(T_i), \quad \widehat{G}_i = \frac{G_i}{L_{\mathrm{raw}}(T_i)}$$

In [3]:
from music_analysis.themes import (
    filter_by_compression_gain, compute_theme_importance,
)
from music_analysis import config

# 压缩增益过滤
strict_types_f, occurrences_f, symmetric_families_f, gains = \
    filter_by_compression_gain(strict_types, occurrences, symmetric_families,
                                theta_g=config.THETA_G)

# 计算主题重要性 A_i
importance = compute_theme_importance(strict_types_f, gains)

# 保存过滤后的数据
import pickle, os
os.makedirs("data", exist_ok=True)
filtered_data = {
    "occurrences": occurrences_f,
    "strict_types": strict_types_f,
    "symmetric_families": symmetric_families_f,
    "gains": gains,
    "importance": importance,
}
with open("data/themes_filtered.pkl", "wb") as f:
    pickle.dump(filtered_data, f)
print("✅ 过滤后主题数据已保存: data/themes_filtered.pkl")

# Top-10 主题重要性
sorted_imp = sorted(importance.items(), key=lambda x: -x[1])[:10]
print(f"\nTop-10 主题重要性 A_i:")
for tid, a in sorted_imp:
    g = gains[tid]
    print(f"  T_{tid}: A={a:.3f}, n={g['n_i']}, L̄={g['L_bar']:.1f}, "
          f"Ĝ={g['G_hat']:.3f}")

# 替换为过滤后的数据供后续使用
occurrences = occurrences_f
strict_types = strict_types_f
symmetric_families = symmetric_families_f
del occurrences_f, strict_types_f, symmetric_families_f


🔍 压缩增益过滤 (θ_G = 0.0)
  过滤前: K = 736 严格类型, Q = 1025 次出现
  被拒绝: 684 个类型 (Ĝ ≤ 0.0)
  过滤后: K = 52 严格类型, Q = 268 次出现
  压缩比: K 52/736 = 7.1%, Q 268/1025 = 26.1%
  保留类型 Ĝ: min=0.083, max=0.409, mean=0.197
  拒绝类型 Ĝ: min=0.000, max=0.000, mean=0.000
✅ 过滤后主题数据已保存: data/themes_filtered.pkl

Top-10 主题重要性 A_i:
  T_2: A=1.017, n=11, L̄=4.0, Ĝ=0.409
  T_4: A=0.959, n=10, L̄=4.0, Ĝ=0.400
  T_7: A=0.824, n=8, L̄=4.0, Ĝ=0.375
  T_8: A=0.824, n=8, L̄=4.0, Ĝ=0.375
  T_9: A=0.824, n=8, L̄=4.0, Ĝ=0.375
  T_15: A=0.743, n=7, L̄=4.0, Ĝ=0.357
  T_16: A=0.743, n=7, L̄=4.0, Ĝ=0.357
  T_1: A=0.602, n=11, L̄=3.0, Ĝ=0.242
  T_37: A=0.563, n=4, L̄=5.0, Ĝ=0.350
  T_3: A=0.560, n=10, L̄=3.0, Ĝ=0.233
